# 10 — Team Composition Analysis

## Business Question
Which champion combinations appear most frequently together, and do certain compositions win more often?

## What This Covers
- Most common duo/trio picks
- Synergy pairs: which champions win more when played together?
- Anti-synergy: which combinations underperform?
- Role frequency analysis (using summoner spells as proxy for role)

In [4]:
import sys
sys.path.insert(0, '../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

from config import *
from data_loader import load_matches, load_champion_map, build_champion_stats
from plot_utils import set_style, save_plot

set_style()
df = load_matches()
champ_map = load_champion_map()
stats = build_champion_stats(df, champ_map)

# Get high-volume champions for synergy analysis (top 20 by games)
top_champs = stats.nlargest(20, 'games')['champ_id'].tolist()
top_names  = {cid: champ_map.get(cid, str(cid)) for cid in top_champs}
print(f"Analysing synergy among top {len(top_champs)} champions by pick rate")
print("Champions:", list(top_names.values()))

Analysing synergy among top 20 champions by pick rate
Champions: ['Thresh', 'Tristana', 'Vayne', 'Kayn', 'Lee Sin', 'Twitch', 'Janna', 'Lucian', 'Jhin', 'Jinx', 'Yasuo', 'Xayah', 'Lux', 'Blitzcrank', 'Rakan', 'Caitlyn', 'Lulu', 'Jax', 'Orianna', 'Zed']


In [5]:
# Build pair synergy matrix
# For each pair of champions that appear on the same team, track win rate
pair_stats = {}

for team in [1, 2]:
    champ_cols = [f't{team}_champ{i}id' for i in range(1, 6)]
    won_col = 't1_won'
    
    for _, row in df.iterrows():
        champs = [int(row[c]) for c in champ_cols if int(row[c]) in top_champs]
        won = row[won_col] if team == 1 else 1 - row[won_col]
        
        for pair in combinations(sorted(champs), 2):
            if pair not in pair_stats:
                pair_stats[pair] = {'games': 0, 'wins': 0}
            pair_stats[pair]['games'] += 1
            pair_stats[pair]['wins'] += won

pair_records = []
for (c1, c2), s in pair_stats.items():
    if s['games'] >= 50:
        pair_records.append({
            'champ1': top_names.get(c1, str(c1)),
            'champ2': top_names.get(c2, str(c2)),
            'games': s['games'],
            'win_rate': round(s['wins'] / s['games'] * 100, 1)
        })

pairs_df = pd.DataFrame(pair_records).sort_values('win_rate', ascending=False)
print(f"\nPairs with 50+ games: {len(pairs_df)}")
print("\nTop 10 best synergy pairs:")
print(pairs_df.head(10).to_string(index=False))
print("\nBottom 10 worst synergy pairs:")
print(pairs_df.tail(10).to_string(index=False))


Pairs with 50+ games: 171

Top 10 best synergy pairs:
  champ1  champ2  games  win_rate
   Janna   Vayne   1014      59.6
  Twitch   Janna    839      59.5
Tristana   Janna   1266      58.8
   Janna Orianna    579      58.2
     Jax   Janna    623      57.9
   Janna   Yasuo    584      57.5
 Caitlyn   Vayne     53      56.6
   Janna     Lux    414      56.3
     Lux    Lulu    335      56.1
  Twitch    Jinx    159      56.0

Bottom 10 worst synergy pairs:
    champ1  champ2  games  win_rate
   Caitlyn     Zed    419      43.4
       Jax Lee Sin    359      43.2
      Lulu  Thresh     59      42.4
    Lucian  Thresh   1132      42.2
   Lee Sin  Lucian    754      41.5
   Lee Sin    Kayn    159      40.9
   Caitlyn Lee Sin    621      39.8
      Jhin  Lucian    142      38.7
   Orianna     Lux    103      37.9
Blitzcrank  Thresh     70      35.7


In [6]:
# Synergy heatmap
pivot_data = []
champ_names = list(top_names.values())

for c1 in champ_names:
    for c2 in champ_names:
        if c1 != c2:
            row = pairs_df[((pairs_df['champ1']==c1) & (pairs_df['champ2']==c2)) |
                           ((pairs_df['champ1']==c2) & (pairs_df['champ2']==c1))]
            wr = row['win_rate'].values[0] if len(row) > 0 else np.nan
        else:
            wr = np.nan
        pivot_data.append({'c1': c1, 'c2': c2, 'wr': wr})

pivot_df = pd.DataFrame(pivot_data).pivot(index='c1', columns='c2', values='wr')

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.isnan(pivot_df.values)
sns.heatmap(pivot_df, annot=True, fmt='.0f', cmap='RdYlGn',
            vmin=40, vmax=65, ax=ax,
            annot_kws={'size': 7}, linewidths=0.3,
            mask=mask)
ax.set_title('Champion Synergy Heatmap\n(Win rate when champions played on same team)')
ax.set_xlabel('')
ax.set_ylabel('')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
save_plot('10_synergy_heatmap.png')
plt.show()

# Bar chart of best/worst pairs
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
top_pairs = pairs_df.nlargest(10, 'win_rate')
bot_pairs = pairs_df.nsmallest(10, 'win_rate')

top_pairs['pair'] = top_pairs['champ1'] + ' + ' + top_pairs['champ2']
bot_pairs['pair'] = bot_pairs['champ1'] + ' + ' + bot_pairs['champ2']

bars_t = axes[0].barh(top_pairs['pair'], top_pairs['win_rate'],
                      color=COLORS['green'], edgecolor='white', height=0.7)
axes[0].axvline(50, color=COLORS['gray'], linestyle='--', linewidth=1.5)
for bar, val in zip(bars_t, top_pairs['win_rate']):
    axes[0].text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                f'{val}%', va='center', fontsize=9)
axes[0].set_xlabel('Win Rate (%)')
axes[0].set_title('Top 10 Best Champion Synergies')

bars_b = axes[1].barh(bot_pairs['pair'], bot_pairs['win_rate'],
                      color=COLORS['red'], edgecolor='white', height=0.7)
axes[1].axvline(50, color=COLORS['gray'], linestyle='--', linewidth=1.5)
for bar, val in zip(bars_b, bot_pairs['win_rate']):
    axes[1].text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                f'{val}%', va='center', fontsize=9)
axes[1].set_xlabel('Win Rate (%)')
axes[1].set_title('Bottom 10 Worst Champion Synergies')

plt.suptitle('Champion Synergy Analysis', fontsize=14, fontweight='bold')
save_plot('10b_synergy_bar.png')
plt.show()

  Saved -> plots/10_synergy_heatmap.png
  Saved -> plots/10b_synergy_bar.png


## Summary

The synergy heatmap reveals which champion combinations perform above their individual win rates — these are the genuine synergies that arise from kit interactions (e.g., engage + follow-up, lockdown + burst damage combos). High-synergy pairs in the top-left corner (both champions are already above 50% individually) represent the meta's strongest compositions. Balance teams should cross-reference these findings with patch notes to understand whether synergies are intentional design or unintended interactions.